# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DipeshGhimire33/Flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am provisionally choosing Refresh / Content Opportunity Scoring. The goal is to investigate whether signals in the available data can help prioritize pages for content review. Rather than treating the task as simply predicting a metric, I want to produce a ranked review queue that connects analysis to an actionable content decision.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. The question: decision, action, cost of a wrong call

Research question: Which pages appear to have the strongest evidence of being content-refresh opportunities, and can the available signals support a useful ranked review queue?

Decision: Which pages should a content/SEO team review first?

Action: Review the highest-priority pages and decide whether each should be refreshed, left unchanged, or investigated further.

Unit of analysis: Page.

Cost of a wrong recommendation: A false positive could waste limited content/SEO effort on a page that does not need refreshing. A false negative could cause the team to overlook a page where an update might have been valuable.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df =pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='str')

In [5]:
print("Unique content items:", df["content_id"].nunique())

Unique content items: 30000


In [6]:
print("Median days since last update:", df["days_since_last_update"].median())

Median days since last update: 20.0


In [7]:
print(df["trend_direction"].value_counts(dropna=False))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [8]:
print(
    "Pages that are both declining and haven't been updated in 180+ days:",
    (
        (df["trend_direction"] == "down") &
        (df["days_since_last_update"] >= 180)
    ).sum()
)

Pages that are both declining and haven't been updated in 180+ days: 82


The starter dataset contains **30,000 unique content items**. The median `days_since_last_update` is **20 days**, so the dataset includes information about how recently content was updated. There are also **16,262 pages with a `down` trend**, compared with 4,388 pages with an `up` trend. In addition, **82 pages are both down-trending and have not been updated for at least 180 days**.

These numbers make Refresh / Content Opportunity Scoring worth investigating because the dataset contains both **content freshness signals** and **recent performance/trend signals**. In particular, the 82 pages that are both down-trending and relatively old could provide a concrete starting point for investigating whether a scoring approach can prioritize pages for human review.

These numbers do not show that refreshing a page will improve its performance. They only show that the dataset contains enough variation in content age and performance trends to make a refresh-opportunity ranking worth exploring.

## 4. Careful words: what I can and can't claim

The `trend_direction` distribution is:

| Trend direction |      Pages |
| --------------- | ---------: |
| down            |     16,262 |
| stable          |      5,962 |
| up              |      4,388 |
| new             |      2,236 |
| flat            |      1,152 |
| **Total**       | **30,000** |

There are also **82 pages that are both down-trending and at least 180 days since their last update**. This is a useful descriptive cohort for investigation, but being old and down-trending does not by itself establish that a page should be refreshed.

### What I Can Claim

* The starter dataset contains **30,000 content items** and includes signals related to content age, freshness, performance, engagement, and recent trends.
* A substantial number of pages have a `down` trend, and **82 pages are both down-trending and at least 180 days since their last update**.
* These signals provide a reasonable basis for investigating whether pages can be **ranked into a useful content-review queue**.
* The purpose of the Opportunity Score is to support **human prioritization**, rather than automatically decide which content should be changed.
* The resulting score can indicate which pages show **stronger evidence of being worth reviewing**, relative to other pages in the dataset.

### What I Can't Claim Yet

* I cannot claim that an old page that is trending down **should definitely be refreshed**.
* I cannot claim that refreshing a page will cause its traffic, clicks, engagement, or other performance metrics to increase.
* I cannot claim that the observed signals are **causal drivers** of performance rather than variables correlated with performance.
* I cannot claim that a scoring system will improve business outcomes until it is evaluated against an appropriate baseline and, ideally, against outcomes from actual content changes.
* I cannot claim that the highest-scoring pages are necessarily the **best pages to refresh**, because important considerations may not be represented in the dataset, including business priorities, strategic importance, content quality, editorial judgment, and operational constraints.

### Appropriate Conclusion

The appropriate claim at this stage is:

> **The available data can be used to identify and prioritize pages that are worth human review for potential content refresh.**

Whether those reviews—and any subsequent content refreshes—actually improve performance is a separate question requiring additional evidence, ideally from observed outcomes following real content changes.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Week 1 Completed